# Quantitative Analysis — Milei Climate Narratives Corpus (v4)

Reproducible notebook for the analysis of `milei_corpus_v4_code_counts.csv` (193 coded speeches, Dimension 1 codebook v4).

**What this notebook does, in order:**
1. Data prep & QA — load the corpus, compute word counts, build a long-format code table
2. Descriptive overview — code distributions, sub-code ranking, zero-code speech profile
3. Time trend — code density by quarter across the presidency
4. Venue & country effects — cross-tabs by `tipo`/`pais` with Kruskal-Wallis tests
   - 4b. Venue groups (6 groups + scope) — speeches, codes, density, dimension mix, tests
5. Climate-focused deep dive — validating the `clima` heuristic, "climate-signature" sub-codes, top speeches
6. Co-occurrence & rhetorical-mode clustering — sub-code correlation matrix + k-means clusters

**Requirements:** `pandas`, `numpy`, `scipy`, `scikit-learn`, `matplotlib`. Install with:
```
pip install pandas numpy scipy scikit-learn matplotlib
```

**Paths:** `BASE_DIR` is auto-detected (the folder containing `analysis/` and `data/`); override with the `MILEI_BASE_DIR` environment variable. Run from inside the project folder.

In [1]:
import os
import re
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib
matplotlib.use('Agg')  # switch to default backend if running interactively, e.g. remove this line in Jupyter
import matplotlib.pyplot as plt

# --- Project root: auto-detected (walks up from the working directory until it finds analysis/ and data/).
# Override with the MILEI_BASE_DIR environment variable if needed. ---
def _find_base():
    if os.environ.get("MILEI_BASE_DIR"):
        return os.environ["MILEI_BASE_DIR"]
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isdir(os.path.join(d, "analysis")) and os.path.isdir(os.path.join(d, "data")):
            return d
        if os.path.dirname(d) == d:
            raise RuntimeError("Project root not found; set MILEI_BASE_DIR")
        d = os.path.dirname(d)
BASE_DIR = _find_base()
print("BASE_DIR =", BASE_DIR)
try:
    display
except NameError:          # allow running as a plain script
    display = print

CORPUS_CSV = os.path.join(BASE_DIR, "analysis/results/v4/milei_corpus_v4_code_counts.csv")
RAW_SPEECHES_DIR = os.path.join(BASE_DIR, "data/raw_speeches")
OUT_DIR = os.path.join(BASE_DIR, "analysis/results/v4/quant_analysis")
CHARTS_DIR = os.path.join(OUT_DIR, "charts")
os.makedirs(CHARTS_DIR, exist_ok=True)

pd.set_option('display.width', 140)
pd.set_option('display.max_rows', 40)

BASE_DIR = /sessions/rcw-01q9enyadwzajmdw1gaeid5c/mnt/milei-climate-narratives


## 1. Data prep & QA

Load the master code-count CSV, confirm all rows are `coded=ok`, and compute a **word count per speech** from the raw (uncoded) transcripts in `data/raw_speeches/` — using the raw files rather than the annotated ones in `analysis/coded_speeches/v4/` avoids inflating counts with `[CODE: ...]` tags. Word counts let us normalize code counts by speech length later (a 300-word cadena nacional clip and a 7,700-word Davos keynote are not comparable on raw counts alone).

In [2]:
df = pd.read_csv(CORPUS_CSV)
print("Rows:", len(df), "| Columns:", len(df.columns))
print("coded == 'ok' for all rows:", (df['coded'] == 'ok').all())

subcodes = [
    'D1.1_Market_Primacy', 'D1.1_Anti_State_Anti_Bureaucratic', 'D1.1_Anti_Casta',
    'D1.1_Delegitimization_of_Regulation', 'D1.1_Sustainability_as_Ideological_or_Harmful',
    'D1.1_Rejection_of_Multilateral_Governance',
    'D1.2_Attack_on_Universities_and_Research_Institutions', 'D1.2_Attack_on_Experts_and_Technocrats',
    'D1.2_Climate_Skepticism_or_Dismissal', 'D1.2_Pseudo_Science_or_Ideological_Science', 'D1.2_Selective_Empiricism',
    'D1.3_Woke_Bundling', 'D1.3_Anti_Socialism_Anti_Collectivism', 'D1.3_Civilizational_Defense',
]

Rows: 193 | Columns: 34
coded == 'ok' for all rows: True


In [3]:
def word_count(fname):
    """Word count from the raw (unannotated) transcript matched via `matched_coded_file`."""
    path = os.path.join(RAW_SPEECHES_DIR, fname)
    if not os.path.exists(path):
        return None
    txt = open(path, encoding='utf-8').read()
    # strip YAML frontmatter (content between the first two '---' markers)
    parts = txt.split('---')
    body = '---'.join(parts[2:]) if len(parts) >= 3 else txt
    return len(re.findall(r"\b\w+\b", body, re.UNICODE))

df['word_count'] = df['matched_coded_file'].apply(word_count)
print("Missing word counts:", df['word_count'].isna().sum())
df['word_count'].describe()

Missing word counts: 0


count      193.000000
mean      3963.435233
std       3112.700560
min        152.000000
25%       1238.000000
50%       3174.000000
75%       6419.000000
max      11702.000000
Name: word_count, dtype: float64

In [4]:
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
print("Date parse failures:", df['fecha'].isna().sum())
print("Date range:", df['fecha'].min().date(), "to", df['fecha'].max().date())

df['year'] = df['fecha'].dt.year
df['quarter'] = df['fecha'].dt.to_period('Q').astype(str)
df['month'] = df['fecha'].dt.to_period('M').astype(str)

# Rates per 1,000 words -- the primary normalized metric used throughout this notebook
df['codes_per_1k'] = df['Codes_Total'] / df['word_count'] * 1000
df['d11_per_1k'] = df['D1.1_Total'] / df['word_count'] * 1000
df['d12_per_1k'] = df['D1.2_Total'] / df['word_count'] * 1000
df['d13_per_1k'] = df['D1.3_Total'] / df['word_count'] * 1000

df.to_csv(os.path.join(OUT_DIR, 'corpus_enriched.csv'), index=False)
df[['fecha', 'quarter', 'word_count', 'Codes_Total', 'codes_per_1k']].head()

Date parse failures: 0
Date range: 2023-10-08 to 2026-04-29


,fecha,quarter,word_count,Codes_Total,codes_per_1k
0,2023-10-08,2023Q4,312,3,9.615385
1,2023-12-10,2023Q4,3446,5,1.450958
2,2023-12-10,2023Q4,354,0,0.000000
3,2023-12-20,2023Q4,2268,7,3.086420
4,2023-12-30,2023Q4,1150,3,2.608696


Build a **long-format table** (one row per code instance) alongside the wide table — this makes sub-code ranking and co-occurrence analysis much simpler than pivoting the wide table each time.

In [5]:
long = df.melt(
    id_vars=['id', 'fecha', 'year', 'quarter', 'pais', 'tipo', 'venue_group', 'scope', 'subtipo', 'clima', 'word_count'],
    value_vars=subcodes, var_name='subcode', value_name='count'
)
long = long[long['count'] > 0].copy()
long['dimension'] = long['subcode'].str.split('_').str[0]
long.to_csv(os.path.join(OUT_DIR, 'long_format.csv'), index=False)
print("Long-format rows (code instances):", len(long))

Long-format rows (code instances): 475


## 2. Descriptive overview

Distribution of total codes per speech, ranking of the 14 sub-codes by frequency, and a profile of the zero-code speeches (do they cluster in a particular genre, or are they scattered randomly?).

In [6]:
print(df['Codes_Total'].describe())
print()
dim_totals = df[['D1.1_Total', 'D1.2_Total', 'D1.3_Total']].sum()
print("Dimension totals:")
print(dim_totals)
print((dim_totals / dim_totals.sum() * 100).round(1), "% share")

count    193.000000
mean       3.093264
std        3.053619
min        0.000000
25%        1.000000
50%        2.000000
75%        4.000000
max       21.000000
Name: Codes_Total, dtype: float64

Dimension totals:
D1.1_Total    375
D1.2_Total     52
D1.3_Total    170
dtype: int64
D1.1_Total    62.8
D1.2_Total     8.7
D1.3_Total    28.5
dtype: float64 % share


In [7]:
sub_totals = long.groupby('subcode')['count'].sum().sort_values(ascending=False)
sub_totals

subcode
D1.1_Anti_Casta                                          116
D1.3_Anti_Socialism_Anti_Collectivism                    102
D1.1_Anti_State_Anti_Bureaucratic                         92
D1.1_Delegitimization_of_Regulation                       84
D1.3_Woke_Bundling                                        41
D1.1_Market_Primacy                                       35
D1.3_Civilizational_Defense                               27
D1.1_Rejection_of_Multilateral_Governance                 24
D1.1_Sustainability_as_Ideological_or_Harmful             24
D1.2_Attack_on_Experts_and_Technocrats                    20
D1.2_Attack_on_Universities_and_Research_Institutions     14
D1.2_Pseudo_Science_or_Ideological_Science                 8
D1.2_Climate_Skepticism_or_Dismissal                       7
D1.2_Selective_Empiricism                                  3
Name: count, dtype: int64

In [8]:
codes_total_stats = df['Codes_Total'].describe().rename_axis('stat').to_frame('value')
dim_totals_df = dim_totals.rename('count').to_frame()
dim_totals_df['share_pct'] = (dim_totals_df['count'] / dim_totals_df['count'].sum() * 100).round(1)

display(codes_total_stats)
print()
print("Dimension totals:")
display(dim_totals_df)

,value
stat,
count,193.000000
mean,3.093264
std,3.053619
min,0.000000
25%,1.000000
50%,2.000000
75%,4.000000
max,21.000000



Dimension totals:


,count,share_pct
D1.1_Total,375,62.8
D1.2_Total,52,8.7
D1.3_Total,170,28.5


In [9]:
zero = df[df['Codes_Total'] == 0]
print("Zero-code speeches:", len(zero), f"({len(zero)/len(df)*100:.0f}% of corpus)")
print()
print("By tipo:")
print(zero['tipo'].value_counts())
print()
print("By pais:")
print(zero['pais'].value_counts())
print()
print("By clima flag:")
print(zero['clima'].fillna('blank').value_counts())

Zero-code speeches: 23 (12% of corpus)

By tipo:
tipo
acto institucional                5
acto conmemorativo                5
bilateral / diplomatico           4
cumbre internacional              3
evento ideologico/politico        2
sectorial                         1
premio / reconocimiento           1
acto institucional / bilateral    1
acto conmemorativo/patrio         1
Name: count, dtype: int64

By pais:
pais
argentina      13
israel          4
ee.uu.          2
el salvador     1
suiza           1
italia          1
paraguay        1
Name: count, dtype: int64

By clima flag:
clima
blank    19
~         4
Name: count, dtype: int64


In [10]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df['Codes_Total'], bins=range(0, 21), color='#4A4A4A', edgecolor='white')
ax.set_xlabel('Codes_Total per speech')
ax.set_ylabel('Number of speeches')
ax.set_title(f"Distribution of D1 code counts per speech (n={len(df)}, {(df.Codes_Total==0).sum()} zero-code)")
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '03_codes_histogram.png'), dpi=150)
plt.show()

In [11]:
short_names = [c.split('_', 1)[1].replace('_', ' ') for c in sub_totals.index]
colors = ['#8B2E2E' if s.startswith('D1.1') else '#2E6B4A' if s.startswith('D1.2') else '#2E4A8B'
          for s in sub_totals.index]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(short_names[::-1], sub_totals.values[::-1], color=colors[::-1])
ax.set_xlabel('Total instances across corpus')
ax.set_title('D1 sub-code frequency ranking\n(red=D1.1 State/Sustainability, green=D1.2 Science/Expertise, blue=D1.3 Cultural Backlash)')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '02_subcode_ranking.png'), dpi=150)
plt.show()

## 4. Venue & country effects

Cross-tabulate code density (`codes_per_1k`) by `venue_group` (venue/speech type) and `pais` (country), and test for group differences with the **Kruskal-Wallis** test (nonparametric — the rate distribution is right-skewed, so ANOVA's normality assumption doesn't hold). Groups below a minimum size are excluded from the test itself since a single-speech "group" doesn't provide a meaningful test of central tendency.

In [12]:
tipo_stats = df.groupby('venue_group').agg(
    n=('id', 'count'),
    mean_codes_per_1k=('codes_per_1k', 'mean'),
    median_codes_per_1k=('codes_per_1k', 'median'),
    mean_word_count=('word_count', 'mean'),
).sort_values('mean_codes_per_1k', ascending=False)
tipo_stats.to_csv(os.path.join(OUT_DIR, 'tipo_stats.csv'))
tipo_stats.round(2)

,n,mean_codes_per_1k,median_codes_per_1k,mean_word_count
venue_group,,,,
International & Diplomatic,26,2.03,1.14,1308.96
Ideological & Political Events,22,1.81,1.12,4206.95
Commemorative & Community,14,0.93,0.80,1371.50
State & Institutional Acts,37,0.91,0.84,2792.00
Economic & Business Forums,61,0.85,0.52,5654.03
"Awards, Academic & Think Tanks",33,0.79,0.64,5180.48


In [13]:
# Kruskal-Wallis across venue groups with n >= 5 (stable estimate threshold)
groups_tipo = [g['codes_per_1k'].values for name, g in df.groupby('venue_group') if len(g) >= 5]
h, p = stats.kruskal(*groups_tipo)
print(f"Kruskal-Wallis across venue groups (k={len(groups_tipo)} groups with n>=5): H={h:.2f}, p={p:.4f}")

Kruskal-Wallis across venue groups (k=6 groups with n>=5): H=7.53, p=0.1844


In [14]:
pais_stats = df.groupby('pais').agg(
    n=('id', 'count'),
    mean_codes_per_1k=('codes_per_1k', 'mean'),
    median_codes_per_1k=('codes_per_1k', 'median'),
).sort_values('mean_codes_per_1k', ascending=False)
pais_stats.to_csv(os.path.join(OUT_DIR, 'pais_stats.csv'))
pais_stats.round(2)

,n,mean_codes_per_1k,median_codes_per_1k
pais,,,
brasil,4,3.22,3.11
ee.uu.,21,2.30,1.24
suiza,5,1.75,0.90
paraguay,5,1.50,0.58
hungria,2,1.48,1.48
españa,8,1.31,1.22
uruguay,3,1.22,0.97
argentina,129,0.88,0.66
israel,10,0.78,0.74


In [15]:
groups_pais = [g['codes_per_1k'].values for name, g in df.groupby('pais') if len(g) >= 3]
h2, p2 = stats.kruskal(*groups_pais)
print(f"Kruskal-Wallis across pais (k={len(groups_pais)} groups with n>=3): H={h2:.2f}, p={p2:.4f}")

# Argentina vs. rest-of-world binary comparison
df['ar_vs_intl'] = np.where(df['pais'] == 'argentina', 'argentina', 'international')
u, pu = stats.mannwhitneyu(
    df.loc[df.ar_vs_intl == 'argentina', 'codes_per_1k'],
    df.loc[df.ar_vs_intl == 'international', 'codes_per_1k'],
)
print(df.groupby('ar_vs_intl')['codes_per_1k'].agg(['count', 'mean', 'median']).round(3))
print(f"Mann-Whitney U={u:.1f}, p={pu:.4f}")

Kruskal-Wallis across pais (k=8 groups with n>=3): H=15.17, p=0.0339
               count   mean  median
ar_vs_intl                         
argentina        129  0.878   0.656
international     64  1.622   0.908
Mann-Whitney U=3456.0, p=0.0658


## 4b. Venue groups (basic statistical overview)

`venue_group` collapses the 23 raw `tipo` values into 6 analytical groups (mapping in `corpus_metadata.TIPO_GROUP`, identical to the *Venue Group* column of the Excel workbook); `scope` is National (pais = argentina) vs International. This section gives, per group: **how many speeches**, how long they are, **how many codes** (total and per speech), how many speeches have no codes, code density (codes per 1,000 words), and the **mix of dimensions** D1.1 / D1.2 / D1.3. Tests: Kruskal-Wallis on code density (with epsilon-squared effect size) and a chi-square test of independence between venue group and dimension (Cramér's V).

In [16]:
GROUPS = ['Economic & Business Forums', 'State & Institutional Acts', 'Awards, Academic & Think Tanks',
          'International & Diplomatic', 'Ideological & Political Events', 'Commemorative & Community']
assert set(df['venue_group']) <= set(GROUPS), set(df['venue_group']) - set(GROUPS)

vg = df.groupby('venue_group').agg(
    speeches=('id', 'count'),
    total_words=('word_count', 'sum'),
    mean_words=('word_count', 'mean'),
    total_codes=('Codes_Total', 'sum'),
    mean_codes_per_speech=('Codes_Total', 'mean'),
    median_codes_per_speech=('Codes_Total', 'median'),
    zero_code_speeches=('Codes_Total', lambda s: int((s == 0).sum())),
    mean_codes_per_1k=('codes_per_1k', 'mean'),
    median_codes_per_1k=('codes_per_1k', 'median'),
    D1_1=('D1.1_Total', 'sum'), D1_2=('D1.2_Total', 'sum'), D1_3=('D1.3_Total', 'sum'),
).reindex(GROUPS)
vg['pct_of_speeches'] = vg['speeches'] / vg['speeches'].sum() * 100
vg['pct_of_codes'] = vg['total_codes'] / vg['total_codes'].sum() * 100
vg['pct_zero_code'] = vg['zero_code_speeches'] / vg['speeches'] * 100
vg['codes_per_1k_pooled'] = vg['total_codes'] / vg['total_words'] * 1000   # total codes / total words
for d in ('D1_1', 'D1_2', 'D1_3'):
    vg[d + '_share_pct'] = vg[d] / vg['total_codes'] * 100
vg.loc['TOTAL'] = [np.nan] * len(vg.columns)
vg.loc['TOTAL', ['speeches', 'total_words', 'total_codes', 'zero_code_speeches', 'D1_1', 'D1_2', 'D1_3']] = \
    vg.loc[GROUPS, ['speeches', 'total_words', 'total_codes', 'zero_code_speeches', 'D1_1', 'D1_2', 'D1_3']].sum()
vg.loc['TOTAL', 'mean_words'] = df['word_count'].mean()
vg.loc['TOTAL', 'mean_codes_per_speech'] = df['Codes_Total'].mean()
vg.loc['TOTAL', 'median_codes_per_speech'] = df['Codes_Total'].median()
vg.loc['TOTAL', 'mean_codes_per_1k'] = df['codes_per_1k'].mean()
vg.loc['TOTAL', 'median_codes_per_1k'] = df['codes_per_1k'].median()
vg.loc['TOTAL', 'pct_of_speeches'] = 100.0
vg.loc['TOTAL', 'pct_of_codes'] = 100.0
vg.loc['TOTAL', 'pct_zero_code'] = vg.loc['TOTAL', 'zero_code_speeches'] / vg.loc['TOTAL', 'speeches'] * 100
vg.loc['TOTAL', 'codes_per_1k_pooled'] = vg.loc['TOTAL', 'total_codes'] / vg.loc['TOTAL', 'total_words'] * 1000
for d in ('D1_1', 'D1_2', 'D1_3'):
    vg.loc['TOTAL', d + '_share_pct'] = vg.loc['TOTAL', d] / vg.loc['TOTAL', 'total_codes'] * 100
vg.to_csv(os.path.join(OUT_DIR, 'venue_group_stats.csv'))
display(vg.round(2))

,speeches,total_words,mean_words,total_codes,mean_codes_per_speech,median_codes_per_speech,zero_code_speeches,mean_codes_per_1k,median_codes_per_1k,D1_1,D1_2,D1_3,pct_of_speeches,pct_of_codes,pct_zero_code,codes_per_1k_pooled,D1_1_share_pct,D1_2_share_pct,D1_3_share_pct
venue_group,,,,,,,,,,,,,,,,,,,
Economic & Business Forums,61.0,344896.0,5654.03,234.0,3.84,3.0,1.0,0.85,0.52,153.0,23.0,58.0,31.61,39.20,1.64,0.68,65.38,9.83,24.79
State & Institutional Acts,37.0,103304.0,2792.00,90.0,2.43,2.0,6.0,0.91,0.84,64.0,7.0,19.0,19.17,15.08,16.22,0.87,71.11,7.78,21.11
"Awards, Academic & Think Tanks",33.0,170956.0,5180.48,98.0,2.97,3.0,1.0,0.79,0.64,58.0,11.0,29.0,17.10,16.42,3.03,0.57,59.18,11.22,29.59
International & Diplomatic,26.0,34033.0,1308.96,62.0,2.38,1.5,7.0,2.03,1.14,43.0,2.0,17.0,13.47,10.39,26.92,1.82,69.35,3.23,27.42
Ideological & Political Events,22.0,92553.0,4206.95,91.0,4.14,4.0,2.0,1.81,1.12,39.0,9.0,43.0,11.40,15.24,9.09,0.98,42.86,9.89,47.25
Commemorative & Community,14.0,19201.0,1371.50,22.0,1.57,1.0,6.0,0.93,0.80,18.0,0.0,4.0,7.25,3.69,42.86,1.15,81.82,0.00,18.18
TOTAL,193.0,764943.0,3963.44,597.0,3.09,2.0,23.0,1.12,0.75,375.0,52.0,170.0,100.00,100.00,11.92,0.78,62.81,8.71,28.48


**Scope within venue group** — number of speeches, National (Argentina) vs International.

In [17]:
vg_scope = pd.crosstab(df['venue_group'], df['scope'], margins=True, margins_name='TOTAL').reindex(GROUPS + ['TOTAL'])
vg_scope_codes = df.pivot_table(index='venue_group', columns='scope', values='Codes_Total', aggfunc='sum',
                                margins=True, margins_name='TOTAL').reindex(GROUPS + ['TOTAL'])
vg_scope.to_csv(os.path.join(OUT_DIR, 'venue_group_by_scope_speeches.csv'))
vg_scope_codes.to_csv(os.path.join(OUT_DIR, 'venue_group_by_scope_codes.csv'))
print("Speeches:"); display(vg_scope)
print("Codes:"); display(vg_scope_codes)

sc = df.groupby('scope').agg(speeches=('id', 'count'), codes=('Codes_Total', 'sum'),
                             mean_codes_per_1k=('codes_per_1k', 'mean'), median_codes_per_1k=('codes_per_1k', 'median'))
display(sc.round(2))

Speeches:


scope,International,National,TOTAL
venue_group,,,
Economic & Business Forums,9,52,61
State & Institutional Acts,2,35,37
"Awards, Academic & Think Tanks",19,14,33
International & Diplomatic,20,6,26
Ideological & Political Events,13,9,22
Commemorative & Community,1,13,14
TOTAL,64,129,193


Codes:


scope,International,National,TOTAL
venue_group,,,
Economic & Business Forums,70,164,234
State & Institutional Acts,1,89,90
"Awards, Academic & Think Tanks",45,53,98
International & Diplomatic,52,10,62
Ideological & Political Events,54,37,91
Commemorative & Community,7,15,22
TOTAL,229,368,597


,speeches,codes,mean_codes_per_1k,median_codes_per_1k
scope,,,,
International,64,229,1.62,0.91
National,129,368,0.88,0.66


**Sub-code composition by venue group** (counts) and **tests**.

In [18]:
comp_vg = df.groupby('venue_group')[subcodes].sum().reindex(GROUPS)
comp_vg.to_csv(os.path.join(OUT_DIR, 'subcode_composition_by_venue_group.csv'))
dim_vg = vg.loc[GROUPS, ['speeches', 'total_codes', 'D1_1', 'D1_2', 'D1_3']].rename(
    columns={'D1_1': 'D1.1_Total', 'D1_2': 'D1.2_Total', 'D1_3': 'D1.3_Total'})
dim_vg.to_csv(os.path.join(OUT_DIR, 'dim_by_venue_group.csv'))
display(comp_vg)

# 1) Kruskal-Wallis on code density across the 6 venue groups
samples = [df.loc[df.venue_group == g, 'codes_per_1k'].values for g in GROUPS]
H, pH = stats.kruskal(*samples)
N, k = len(df), len(GROUPS)
eps2 = (H - k + 1) / (N - k)       # epsilon-squared effect size
print(f"Kruskal-Wallis, codes_per_1k across {k} venue groups: H={H:.2f}, p={pH:.4f}, epsilon^2={eps2:.3f}")

# 2) Chi-square: venue group x dimension (code counts); Cramer's V
ct = dim_vg[['D1.1_Total', 'D1.2_Total', 'D1.3_Total']]
chi2, pchi, dof, _ = stats.chi2_contingency(ct)
V = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
print(f"Chi-square venue_group x dimension: chi2={chi2:.2f}, dof={dof}, p={pchi:.4f}, Cramer's V={V:.3f}")
print("(Code instances are not independent observations - treat the chi-square as descriptive.)")

# 3) National vs International (code density)
a = df.loc[df.scope == 'National', 'codes_per_1k']; b = df.loc[df.scope == 'International', 'codes_per_1k']
u2, pu2 = stats.mannwhitneyu(a, b)
print(f"Mann-Whitney scope (National vs International) on codes_per_1k: U={u2:.1f}, p={pu2:.4f}")

,D1.1_Market_Primacy,D1.1_Anti_State_Anti_Bureaucratic,D1.1_Anti_Casta,D1.1_Delegitimization_of_Regulation,D1.1_Sustainability_as_Ideological_or_Harmful,D1.1_Rejection_of_Multilateral_Governance,D1.2_Attack_on_Universities_and_Research_Institutions,D1.2_Attack_on_Experts_and_Technocrats,D1.2_Climate_Skepticism_or_Dismissal,D1.2_Pseudo_Science_or_Ideological_Science,D1.2_Selective_Empiricism,D1.3_Woke_Bundling,D1.3_Anti_Socialism_Anti_Collectivism,D1.3_Civilizational_Defense
venue_group,,,,,,,,,,,,,,
Economic & Business Forums,17,37,49,38,8,4,4,9,2,5,3,12,38,8
State & Institutional Acts,5,16,25,13,3,2,5,2,0,0,0,5,13,1
"Awards, Academic & Think Tanks",4,16,16,19,3,0,2,5,1,3,0,4,23,2
International & Diplomatic,2,10,3,9,5,14,0,2,0,0,0,7,4,6
Ideological & Political Events,6,6,14,4,5,4,3,2,4,0,0,12,23,8
Commemorative & Community,1,7,9,1,0,0,0,0,0,0,0,1,1,2


Kruskal-Wallis, codes_per_1k across 6 venue groups: H=7.53, p=0.1844, epsilon^2=0.014
Chi-square venue_group x dimension: chi2=29.22, dof=10, p=0.0011, Cramer's V=0.156
(Code instances are not independent observations - treat the chi-square as descriptive.)
Mann-Whitney scope (National vs International) on codes_per_1k: U=3456.0, p=0.0658


In [19]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
short = {g: g.replace('Awards, Academic & Think Tanks', 'Awards & Academic').replace('Economic & Business Forums', 'Economic & Business')
          .replace('State & Institutional Acts', 'State & Institutional').replace('Ideological & Political Events', 'Ideological & Political')
          .replace('International & Diplomatic', 'Intl & Diplomatic').replace('Commemorative & Community', 'Commemorative') for g in GROUPS}
lab = [short[g] for g in GROUPS]
axes[0].barh(lab[::-1], vg.loc[GROUPS, 'speeches'].values[::-1], color='#4A4A4A')
for i, v in enumerate(vg.loc[GROUPS, 'speeches'].values[::-1]):
    axes[0].text(v + 0.5, i, f"{int(v)} ({v/len(df)*100:.0f}%)", va='center', fontsize=9)
axes[0].set_xlabel('Number of speeches'); axes[0].set_title('Speeches by venue group'); axes[0].set_xlim(0, vg.loc[GROUPS,'speeches'].max()*1.25)

shares = vg.loc[GROUPS, ['D1_1_share_pct', 'D1_2_share_pct', 'D1_3_share_pct']].values
left = np.zeros(len(GROUPS))
for j, (nm, col) in enumerate(zip(['D1.1 State/Sustainability', 'D1.2 Science/Expertise', 'D1.3 Cultural Backlash'],
                                  ['#8B2E2E', '#2E6B4A', '#2E4A8B'])):
    axes[1].barh(lab[::-1], shares[::-1, j], left=left[::-1], color=col, label=nm)
    left += shares[:, j]
axes[1].set_xlabel('% of codes in the venue group'); axes[1].set_title('Dimension mix by venue group'); axes[1].legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '11_venue_group_overview.png'), dpi=150)
plt.show()

## 4c. Dimension and sub-code composition by venue type (`tipo`) and country (`pais`)

Beyond density, the **mix** of D1.1 / D1.2 / D1.3 codes (and the 14 sub-codes) can shift with context. Tables show each group's codes as % of that group's total codes. Groups: `tipo` with n >= 5 speeches, `pais` with n >= 3.

In [20]:
def composition(by, min_n, cols):
    g = df.groupby(by)
    n = g['id'].count()
    keep = n[n >= min_n].index
    sums = g[cols].sum().loc[keep]
    tot = sums.sum(axis=1)
    pct = sums.div(tot.replace(0, np.nan), axis=0) * 100
    return pct, n.loc[keep], tot

dimcols = ['D1.1_Total', 'D1.2_Total', 'D1.3_Total']
for by, mn in (('tipo', 5), ('pais', 3)):
    pct, n, tot = composition(by, mn, dimcols)
    out = pct.assign(n=n, total_codes=tot).sort_values('total_codes', ascending=False)
    out.to_csv(os.path.join(OUT_DIR, f'dim_by_{by}.csv'))
    pct_sub, _, _ = composition(by, mn, subcodes)
    pct_sub = pct_sub.dropna().loc[out.index.intersection(pct_sub.dropna().index)]
    pct_sub.round(1).to_csv(os.path.join(OUT_DIR, f'subcode_composition_by_{by}.csv'))
    print(f"--- dimension mix (%) by {by} ---"); display(out.round(1))

--- dimension mix (%) by tipo ---


,D1.1_Total,D1.2_Total,D1.3_Total,n,total_codes
tipo,,,,,
foro economico,59.8,11.6,28.6,21,112
evento ideologico/politico,43.2,8.0,48.9,21,88
acto institucional,66.1,10.2,23.7,26,59
fundacion / think tank,57.4,12.8,29.8,13,47
cumbre internacional,80.0,5.0,15.0,12,40
bolsa / camara de comercio,67.7,6.5,25.8,12,31
cadena nacional,80.6,3.2,16.1,10,31
empresa,78.6,10.7,10.7,10,28
premio / reconocimiento,46.4,14.3,39.3,14,28


--- dimension mix (%) by pais ---


,D1.1_Total,D1.2_Total,D1.3_Total,n,total_codes
pais,,,,,
argentina,65.2,9.8,25.0,129,368
ee.uu.,63.0,4.3,32.6,21,92
españa,48.9,4.4,46.7,8,45
suiza,62.5,21.9,15.6,5,32
israel,64.7,0.0,35.3,10,17
brasil,62.5,6.2,31.2,4,16
paraguay,44.4,0.0,55.6,5,9
uruguay,66.7,0.0,33.3,3,6


In [21]:
def short_sub(c): return c.split('_', 1)[1].replace('_', ' ')
dimcols_pct = {}
for by, mn, fn in (('tipo', 5, '08_subcode_composition_by_tipo.png'), ('pais', 3, '09_subcode_composition_by_pais.png')):
    pct_sub = pd.read_csv(os.path.join(OUT_DIR, f'subcode_composition_by_{by}.csv'), index_col=0)
    fig, ax = plt.subplots(figsize=(12, 0.45 * len(pct_sub) + 3))
    im = ax.imshow(pct_sub.values, cmap='Reds', aspect='auto')
    ax.set_xticks(range(len(pct_sub.columns))); ax.set_xticklabels([short_sub(c) for c in pct_sub.columns], rotation=90)
    ax.set_yticks(range(len(pct_sub.index))); ax.set_yticklabels(pct_sub.index)
    for r in range(pct_sub.shape[0]):
        for k in range(pct_sub.shape[1]):
            v = pct_sub.values[r, k]
            if v >= 1: ax.text(k, r, f"{v:.0f}", ha='center', va='center', fontsize=7)
    plt.colorbar(im, label='% of group codes'); ax.set_title(f'Sub-code composition by {by} (% of the group\'s codes)')
    plt.tight_layout(); plt.savefig(os.path.join(CHARTS_DIR, fn), dpi=150); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, by in zip(axes, ('tipo', 'pais')):
    d = pd.read_csv(os.path.join(OUT_DIR, f'dim_by_{by}.csv'), index_col=0)[dimcols].iloc[::-1]
    left = np.zeros(len(d))
    for col, c, nm in zip(dimcols, ['#8B2E2E', '#2E6B4A', '#2E4A8B'], ['D1.1', 'D1.2', 'D1.3']):
        ax.barh(d.index, d[col], left=left, color=c, label=nm); left += d[col].values
    ax.set_xlabel('% of codes'); ax.set_title(f'Dimension mix by {by}'); ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=3, frameon=False)
plt.tight_layout(); plt.savefig(os.path.join(CHARTS_DIR, '10_dimension_composition.png'), dpi=150); plt.show()

## 5. Climate-focused deep dive

Two questions here:

1. **Does the corpus's `clima` selection heuristic (✔ / ~ / blank) actually predict code density?** If yes, that validates using it as a speech-selection filter.
2. **Does climate-obstruction rhetoric bleed into speeches *not* flagged as climate-relevant?** Defined via three "climate-signature" sub-codes: `Sustainability_as_Ideological_or_Harmful`, `Rejection_of_Multilateral_Governance`, `Climate_Skepticism_or_Dismissal`. If a large share of these instances falls outside `clima=✔` speeches, that supports coding climate-adjacent rhetoric wherever it appears rather than gating on an explicit climate mention per passage.

In [22]:
df['clima_f'] = df['clima'].fillna('blank')

profile = df.groupby('clima_f').agg(
    n=('id', 'count'),
    mean_codes_per_1k=('codes_per_1k', 'mean'),
    mean_d11=('d11_per_1k', 'mean'),
    mean_d12=('d12_per_1k', 'mean'),
    mean_d13=('d13_per_1k', 'mean'),
)
print(profile.round(3))

groups_clima = [g['codes_per_1k'].values for name, g in df.groupby('clima_f')]
h3, p3 = stats.kruskal(*groups_clima)
print(f"\nKruskal-Wallis across clima groups: H={h3:.2f}, p={p3:.4f}")

           n  mean_codes_per_1k  mean_d11  mean_d12  mean_d13
clima_f                                                      
blank     72              1.028     0.664     0.062     0.302
~        111              0.854     0.505     0.043     0.306
✔         10              4.824     3.117     1.037     0.670

Kruskal-Wallis across clima groups: H=17.25, p=0.0002


In [23]:
fig, ax = plt.subplots(figsize=(7, 5))
order = ['blank', '~', '✔']
vals = [df.loc[df.clima_f == c, 'codes_per_1k'].mean() for c in order]
labels = ['No climate\nrelevance flagged', 'Possible climate\nrelevance (~)', 'High climate\nrelevance (✔)']
ax.bar(labels, vals, color=['#999999', '#D9A441', '#8B2E2E'])
ax.set_ylabel('Mean D1 codes per 1,000 words')
ax.set_title(f'D1 code density by climate-relevance heuristic (Kruskal-Wallis p={p3:.4f})')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '04_clima_density.png'), dpi=150)
plt.show()

In [24]:
sig_cols = [
    'D1.1_Sustainability_as_Ideological_or_Harmful',
    'D1.1_Rejection_of_Multilateral_Governance',
    'D1.2_Climate_Skepticism_or_Dismissal',
]
df['climate_signature_total'] = df[sig_cols].sum(axis=1)
df['climate_signature_per_1k'] = df['climate_signature_total'] / df['word_count'] * 1000

inside = df.loc[df.clima_f == '✔', 'climate_signature_total'].sum()
outside = df.loc[df.clima_f != '✔', 'climate_signature_total'].sum()
total = inside + outside
print(f"Climate-signature instances inside clima=✔ speeches: {inside} ({inside/total*100:.0f}%)")
print(f"Climate-signature instances outside clima=✔ speeches: {outside} ({outside/total*100:.0f}%)")
print(f"Total: {total}")

df.to_csv(os.path.join(OUT_DIR, 'corpus_enriched.csv'), index=False)  # re-save with the new columns
df.to_csv(os.path.join(OUT_DIR, 'milei_corpus_v4_data_analysis.csv'), index=False)  # same table, name used in earlier drafts

Climate-signature instances inside clima=✔ speeches: 23 (42%)
Climate-signature instances outside clima=✔ speeches: 32 (58%)
Total: 55


In [25]:
top = (
    df[df['climate_signature_total'] > 0]
    .sort_values('climate_signature_per_1k', ascending=False)
    [['fecha', 'evento-discurso', 'pais', 'tipo', 'clima_f', 'climate_signature_total', 'word_count', 'climate_signature_per_1k']]
    .head(15)
)
top

,fecha,evento-discurso,pais,tipo,clima_f,climate_signature_total,word_count,climate_signature_per_1k
58,2024-09-24,79 asamblea general de la onu – debate general,ee.uu.,cumbre internacional,✔,8,1067,7.497657
0,2023-10-08,debate presidencial 2023,argentina,debate presidencial,✔,2,312,6.410256
144,2025-09-24,80 asamblea general de la onu,ee.uu.,cumbre internacional,✔,5,1654,3.022975
75,2024-11-18,cumbre de lideres del g20 – sesion ii,brasil,cumbre internacional,✔,2,770,2.597403
74,2024-11-18,cumbre de lideres del g20 – intervencion,brasil,cumbre internacional,✔,1,876,1.141553
92,2024-12-21,anuncios en materia nuclear (plan nuclear arge...,argentina,acto institucional,✔,1,1238,0.807754
180,2026-03-21,cpac budapest,hungria,evento ideologico/politico,~,3,3742,0.801710
117,2025-06-12,memorando de la democracia y la libertad (con ...,israel,bilateral / diplomatico,blank,1,1334,0.749625
46,2024-07-28,"exposicion de ganaderia, agricultura e industr...",argentina,sectorial,~,3,4299,0.697837
96,2025-01-23,foro economico mundial de davos 2025,suiza,foro economico,✔,3,4456,0.673250


In [26]:
q_sig = df.groupby('quarter').agg(sig_sum=('climate_signature_total', 'sum')).reset_index()
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(q_sig['quarter'], q_sig['sig_sum'], color='#2E6B4A')
ax.set_ylabel('Climate-signature code instances')
ax.set_title('Climate-signature codes (Sustainability_as_Ideological, Rejection_of_Multilateral_Gov, Climate_Skepticism) by quarter')
ax.tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '05_climate_signature_time.png'), dpi=150)
plt.show()

## 6. Co-occurrence and rhetorical-mode clustering

**Co-occurrence:** phi coefficients (Pearson correlation on binary presence/absence) across the 14 sub-codes — which codes tend to appear together within the same speech?

**Clustering:** k-means on each speech's code-**composition** profile (i.e. what % of that speech's codes belong to each sub-code — the "shape" of its rhetorical mix, independent of how many codes it has overall). Composition is used instead of raw rate-per-word because very short, high-rate outlier speeches (e.g. a 300-word debate clip) otherwise dominate a rate-based clustering. Only speeches with `Codes_Total >= 3` are included so each profile is estimated from enough data points to be stable.

In [27]:
X_bin = (df[subcodes] > 0).astype(int)
corr = X_bin.corr()

fig, ax = plt.subplots(figsize=(11, 10))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(short_names))); ax.set_xticklabels(short_names, rotation=90)
ax.set_yticks(range(len(short_names))); ax.set_yticklabels(short_names)
plt.colorbar(im, label='Phi coefficient (co-occurrence correlation)')
ax.set_title(f'Sub-code co-occurrence correlation (presence/absence per speech, n={len(df)})')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '06_cooccurrence_heatmap.png'), dpi=150)
plt.show()

In [28]:
pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().sort_values(ascending=False)
print("Top 10 co-occurring sub-code pairs:")
pairs.head(10)

Top 10 co-occurring sub-code pairs:


D1.2_Pseudo_Science_or_Ideological_Science             D1.2_Selective_Empiricism                        0.423692
D1.1_Rejection_of_Multilateral_Governance              D1.3_Woke_Bundling                               0.402011
D1.2_Attack_on_Universities_and_Research_Institutions  D1.2_Climate_Skepticism_or_Dismissal             0.266286
D1.1_Sustainability_as_Ideological_or_Harmful          D1.2_Attack_on_Experts_and_Technocrats           0.241718
D1.1_Delegitimization_of_Regulation                    D1.2_Pseudo_Science_or_Ideological_Science       0.232205
D1.1_Market_Primacy                                    D1.1_Delegitimization_of_Regulation              0.230436
D1.1_Rejection_of_Multilateral_Governance              D1.3_Civilizational_Defense                      0.228851
D1.1_Sustainability_as_Ideological_or_Harmful          D1.2_Climate_Skepticism_or_Dismissal             0.223685
D1.1_Market_Primacy                                    D1.1_Anti_Casta                          

In [29]:
sub = df[df['Codes_Total'] >= 3].copy()
comp = sub[subcodes].div(sub['Codes_Total'], axis=0)  # composition: each subcode's share of that speech's total codes
X = StandardScaler().fit_transform(comp)

print(f"n speeches with Codes_Total>=3: {len(sub)}\n")
for k in range(2, 7):
    km_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    score = silhouette_score(X, km_k.labels_)
    print(f"k={k}: silhouette={score:.3f}, cluster sizes={np.bincount(km_k.labels_)}")

n speeches with Codes_Total>=3: 92

k=2: silhouette=0.118, cluster sizes=[24 68]
k=3: silhouette=0.146, cluster sizes=[55 28  9]
k=4: silhouette=0.125, cluster sizes=[53  7  6 26]
k=5: silhouette=0.139, cluster sizes=[29 25 14 16  8]
k=6: silhouette=0.173, cluster sizes=[ 7 12 15  3 47  8]


In [30]:
# k=4 chosen: interpretable clusters at a reasonable silhouette score for compositional political-rhetoric data
K = 4
km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(X)
sub['cluster'] = km.labels_

for c in sorted(sub['cluster'].unique()):
    grp = sub[sub['cluster'] == c]
    print(f"--- Cluster {c} (n={len(grp)}, mean codes/1k={grp['codes_per_1k'].mean():.2f}) ---")
    top_codes = comp.loc[grp.index].mean().sort_values(ascending=False).head(4)
    print((top_codes * 100).round(1).astype(str) + '%')
    print('tipo top3:', grp['tipo'].value_counts().head(3).to_dict())
    print('pais top3:', grp['pais'].value_counts().head(3).to_dict())
    print()

sub.to_csv(os.path.join(OUT_DIR, 'clustered_speeches.csv'), index=False)

--- Cluster 0 (n=53, mean codes/1k=1.26) ---
D1.1_Anti_Casta                          25.5%
D1.3_Anti_Socialism_Anti_Collectivism    18.4%
D1.1_Delegitimization_of_Regulation      18.0%
D1.1_Anti_State_Anti_Bureaucratic        17.3%
dtype: object
tipo top3: {'foro economico': 11, 'acto institucional': 8, 'fundacion / think tank': 6}
pais top3: {'argentina': 41, 'ee.uu.': 5, 'españa': 3}

--- Cluster 1 (n=7, mean codes/1k=4.11) ---
D1.1_Rejection_of_Multilateral_Governance    46.0%
D1.3_Woke_Bundling                           18.2%
D1.1_Delegitimization_of_Regulation          14.5%
D1.1_Anti_State_Anti_Bureaucratic             9.0%
dtype: object
tipo top3: {'cumbre internacional': 4, 'bolsa / camara de comercio': 1, 'bilateral / diplomatico': 1}
pais top3: {'ee.uu.': 3, 'argentina': 2, 'brasil': 1}

--- Cluster 2 (n=6, mean codes/1k=2.55) ---
D1.1_Sustainability_as_Ideological_or_Harmful    29.1%
D1.1_Delegitimization_of_Regulation              15.6%
D1.2_Climate_Skepticism_or_Dismissal

In [31]:
# Where do the clusters live? Row-% of each venue / country falling in each cluster (speeches with Codes_Total >= 3)
for by in ('venue_group', 'tipo', 'pais'):
    ct = pd.crosstab(sub[by], sub['cluster'])
    ct.columns = [f'cluster_{c}' for c in ct.columns]
    ct['n'] = ct.sum(axis=1)
    ct.to_csv(os.path.join(OUT_DIR, f'cluster_by_{by}.csv'))
    print(f"--- clusters by {by} (counts) ---"); display(ct)

--- clusters by venue_group (counts) ---


,cluster_0,cluster_1,cluster_2,cluster_3,n
venue_group,,,,,
"Awards, Academic & Think Tanks",13,0,1,4,18
Commemorative & Community,1,0,0,1,2
Economic & Business Forums,22,1,4,10,37
Ideological & Political Events,4,1,1,8,14
International & Diplomatic,1,5,0,2,8
State & Institutional Acts,12,0,0,1,13


--- clusters by tipo (counts) ---


,cluster_0,cluster_1,cluster_2,cluster_3,n
tipo,,,,,
acto institucional,8,0,0,0,8
bilateral / diplomatico,0,1,0,1,2
bolsa / camara de comercio,2,1,1,2,6
cadena nacional,4,0,0,1,5
clase magistral / universidad,3,0,0,0,3
comunitario / religioso,1,0,0,1,2
consejo empresarial,3,0,0,1,4
cumbre internacional,0,4,0,1,5
debate presidencial,0,0,1,0,1


--- clusters by pais (counts) ---


,cluster_0,cluster_1,cluster_2,cluster_3,n
pais,,,,,
argentina,41,2,5,14,62
brasil,0,1,0,2,3
ee.uu.,5,3,0,4,12
españa,3,0,0,3,6
hungria,0,1,0,0,1
israel,2,0,0,1,3
paraguay,0,0,0,1,1
suiza,2,0,1,0,3
uruguay,0,0,0,1,1


In [32]:
pca = PCA(n_components=2).fit(X)
coords = pca.transform(X)
sub['pc1'], sub['pc2'] = coords[:, 0], coords[:, 1]

cluster_labels = {0: 'Cluster 0', 1: 'Cluster 1', 2: 'Cluster 2', 3: 'Cluster 3'}  # relabel descriptively after inspecting means above
colors = {0: '#8B2E2E', 1: '#2E4A8B', 2: '#2E6B4A', 3: '#999999'}

fig, ax = plt.subplots(figsize=(9, 7))
for c in sorted(sub['cluster'].unique()):
    grp = sub[sub['cluster'] == c]
    ax.scatter(grp['pc1'], grp['pc2'], label=f"{cluster_labels[c]} (n={len(grp)})", color=colors[c], s=60, alpha=0.8)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.0f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.0f}% var)')
ax.set_title(f'Rhetorical-mode clusters by D1 code composition (Codes_Total>=3, n={len(sub)})')
ax.legend(fontsize=8, loc='best')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '07_cluster_pca.png'), dpi=150)
plt.show()

## Done

All intermediate tables (`corpus_enriched.csv`, `long_format.csv`, `tipo_stats.csv`, `pais_stats.csv`, `venue_group_stats.csv` (+ `venue_group_by_scope_*.csv`, `dim_by_venue_group.csv`, `subcode_composition_by_venue_group.csv`), `clustered_speeches.csv`) and all 8 charts are written to `analysis/results/v4/quant_analysis/`. See `quant_analysis_memo.md` in the same folder for the full write-up of findings and their implications for the paper draft.